# Market Data Loader — Yahoo Finance & Binance

In [ ]:
# !pip install yfinance pandas requests "websockets>=17" 
# !pip install --upgrade mplfinance



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Data source interface

Common shape both feeds implement: `history()` returns an OHLCV `DataFrame`, `stream()` blocks pushing normalized tick dicts until told to stop.

In [3]:
from __future__ import annotations

import json
import threading
from abc import ABC, abstractmethod
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import Callable

import pandas as pd
import requests
import yfinance as yf
from websockets.sync.client import connect as ws_connect


@dataclass
class DataRequest:
    """One symbol to load/stream. `timeframe` uses Binance-style tokens
    (1m, 5m, 15m, 1h, 4h, 1d, ...); YahooFeed maps/resamples as needed."""
    source: str       # "yahoo" | "binance"
    symbol: str        # "AAPL" | "BTCUSDT"
    timeframe: str = "1h"


def _parse_date(date_str: str) -> pd.Timestamp:
    """Parse "YYYYMMDD" -> UTC Timestamp."""
    return pd.Timestamp(date_str, tz="UTC")


class DataFeed(ABC):
    """Common interface every market data source implements."""

    @abstractmethod
    def history(self, symbol: str, timeframe: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
        """Return OHLCV DataFrame indexed by UTC datetime."""

    @abstractmethod
    def stream(self, symbols: list[str], timeframe: str, on_tick: Callable[[dict], None], stop: threading.Event) -> None:
        """Block, pushing normalized tick/kline dicts to `on_tick` until `stop` is set."""


## Yahoo Finance feed

History via `yfinance`; realtime via its built-in `yf.WebSocket`. yfinance has no native 4h/3d bar, so those are fetched at the closest supported bar and resampled.

In [4]:
class YahooFeed(DataFeed):
    _RESAMPLE_FALLBACK = {"4h": ("1h", "4h"), "3d": ("1d", "3D"), "1w": ("1d", "1W")}

    def history(self, symbol: str, timeframe: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
        fetch_tf, resample_rule = self._RESAMPLE_FALLBACK.get(timeframe, (timeframe, None))
        df = yf.Ticker(symbol).history(start=start, end=end, interval=fetch_tf, auto_adjust=False)
        df = df.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]]
        df.index = pd.to_datetime(df.index, utc=True)
        if resample_rule:
            df = df.resample(resample_rule).agg(
                {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
            ).dropna()
        return df

    def stream(self, symbols, timeframe, on_tick, stop):
        ws = yf.WebSocket(verbose=False)
        ws.subscribe(symbols)

        def handler(msg: dict):
            on_tick({"source": "yahoo", **msg})
            if stop.is_set():
                raise KeyboardInterrupt  # yfinance's listen() loop exits on this

        try:
            ws.listen(handler)
        finally:
            ws.close()


## Binance feed

History via the public REST klines endpoint (paginated 1000 candles/request, no key needed); realtime via Binance's combined kline WebSocket stream.

In [5]:
class BinanceFeed(DataFeed):
    _REST = "https://api.binance.com/api/v3/klines"
    _WS = "wss://stream.binance.com:9443/stream"
    _COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
             "quote_volume", "trades", "taker_base_vol", "taker_quote_vol", "ignore"]

    def history(self, symbol: str, timeframe: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
        start_ms, end_ms = int(start.timestamp() * 1000), int(end.timestamp() * 1000)
        frames = []
        while start_ms < end_ms:
            resp = requests.get(self._REST, params={
                "symbol": symbol, "interval": timeframe,
                "startTime": start_ms, "endTime": end_ms, "limit": 1000,
            }, timeout=30)
            resp.raise_for_status()
            batch = resp.json()
            if not batch:
                break
            frames.append(pd.DataFrame(batch, columns=self._COLS))
            start_ms = batch[-1][6] + 1  # next candle after last close_time

        if not frames:
            return pd.DataFrame(columns=["open", "high", "low", "close", "volume"])

        df = pd.concat(frames, ignore_index=True)
        df.index = pd.to_datetime(df["open_time"], unit="ms", utc=True)
        return df[["open", "high", "low", "close", "volume"]].astype(float)

    def stream(self, symbols, timeframe, on_tick, stop):
        streams = "/".join(f"{s.lower()}@kline_{timeframe}" for s in symbols)
        with ws_connect(f"{self._WS}?streams={streams}") as ws:
            while not stop.is_set():
                try:
                    msg = json.loads(ws.recv(timeout=1))
                except TimeoutError:
                    continue
                on_tick({"source": "binance", **msg["data"]})


## Facade

`MarketDataLoader.load_history` fetches every requested symbol concurrently (any mix of sources/timeframes) into one wide `DataFrame` with `MultiIndex` columns `(symbol, field)`. `stream` runs one background thread per source and normalizes ticks into a single callback.

In [6]:
class MarketDataLoader:
    """Load history / stream ticks from Yahoo Finance and Binance behind one API."""

    _FEEDS: dict[str, DataFeed] = {"yahoo": YahooFeed(), "binance": BinanceFeed()}

    def load_history(self, requests_: list[DataRequest], start: str, end: str) -> pd.DataFrame:
        """
        start, end: "YYYYMMDD"
        Returns one DataFrame, columns = MultiIndex (symbol, field).
        """
        start_ts, end_ts = _parse_date(start), _parse_date(end)

        with ThreadPoolExecutor(max_workers=len(requests_) or 1) as pool:
            results = list(pool.map(
                lambda r: self._FEEDS[r.source].history(r.symbol, r.timeframe, start_ts, end_ts),
                requests_,
            ))

        return pd.concat(results, axis=1, keys=[r.symbol for r in requests_])

    def stream(self, requests_: list[DataRequest], on_tick: Callable[[dict], None]) -> "StreamHandle":
        """Start one background thread per source; normalized ticks go to `on_tick`."""
        stop = threading.Event()
        by_source: dict[str, list[DataRequest]] = {}
        for r in requests_:
            by_source.setdefault(r.source, []).append(r)

        threads = []
        for source, reqs in by_source.items():
            feed = self._FEEDS[source]
            symbols = [r.symbol for r in reqs]
            timeframe = reqs[0].timeframe  # one timeframe per source per stream call
            t = threading.Thread(target=feed.stream, args=(symbols, timeframe, on_tick, stop), daemon=True)
            t.start()
            threads.append(t)

        return StreamHandle(stop, threads)


@dataclass
class StreamHandle:
    _stop: threading.Event
    _threads: list[threading.Thread]

    def stop(self, timeout: float = 5.0) -> None:
        self._stop.set()
        for t in self._threads:
            t.join(timeout=timeout)


## Example — history, multi-source in one call

In [ ]:
loader = MarketDataLoader()

requests_ = [
    DataRequest(source="yahoo", symbol="AAPL", timeframe="4h"),
    DataRequest(source="binance", symbol="BTCUSDT", timeframe="1h"),
]

data = loader.load_history(requests_, start="20260801", end="20260815")
data["AAPL"].tail()


## Example — realtime stream

In [ ]:
ticks = []

handle = loader.stream(
    [DataRequest("yahoo", "AAPL", "1m"), DataRequest("binance", "BTCUSDT", "1m")],
    on_tick=lambda t: ticks.append(t),
)

# ... let it run, then:
# handle.stop()


# Volume

In [9]:
loader = MarketDataLoader()

requests_ = [
    DataRequest(source="binance", symbol="BTCUSDT", timeframe="4h"),
]

data = loader.load_history(requests_, start="20260101", end="20260820")
data["BTCUSDT"].tail()

,open,high,low,close,volume
open_time,,,,,
2026-08-19 08:00:00+00:00,64296.01,64539.05,64255.00,64515.63,1158.29773
2026-08-19 12:00:00+00:00,64515.62,69500.00,64474.86,68554.00,14033.00336
2026-08-19 16:00:00+00:00,68554.00,68997.00,67825.03,68429.44,5078.58687
2026-08-19 20:00:00+00:00,68429.44,70000.00,68390.00,69334.79,5593.23699
2026-08-20 00:00:00+00:00,69334.78,69894.50,68902.22,69195.64,4128.11593
